In [ ]:
import pandas as pd
import pymorphy2
import requests
import spacy
import json
import time
import re

import nltk
from nltk.corpus import framenet as fn
from nltk.corpus.reader.framenet import PrettyList
from nltk.tokenize import sent_tokenize, word_tokenize
from openai import OpenAI
from tqdm import tqdm

In [ ]:
frames = list()
for frame in PrettyList(fn.frames()):
    frame_str = str()
    frame_str += f"{frame.name}"
    # frame_str += f"Frame:\n{frame.name}"
    # frame_str += f"\nDefinition:\n{frame.definition}"
    # frame_str += f"\nFrame Elements:\n"
    # num_fes = 1
    # for fe in frame.FE.keys():
    #     frame_str += f"{num_fes}. {fe}\n"
    #     num_fes += 1
    frame_str += '\n'
    frames.append(frame_str)

len(frames)

In [ ]:
for frame in PrettyList(fn.frames()):
    print(frame.frameRelations)
    break

In [ ]:
print(''.join(frames))

In [ ]:
morph = pymorphy2.MorphAnalyzer()

In [ ]:
stopwords = []

In [ ]:
forum_df = pd.read_csv("../../corpora/ru_forum_headlines/processed_forum_headlines.csv", header=None)
forum = forum_df[0].tolist()

len(forum)

In [ ]:
forum_texts = list()
for text in forum[:30]:
    for word in word_tokenize(text.lower()):
        if morph.parse(word)[0].normal_form in stopwords:
            break
    else:
        forum_texts.append(text)

forum_texts = forum_texts[:10]

forum_texts

In [ ]:
with open("../../corpora/ru_russian_literature/archive/prose/Chekhov/Вишнёвый сад.txt", 'r', encoding="utf-8") as f:
    prose = f.readlines()[21:]

prose_texts = list()
for s in prose:
    new_s = s.strip().rstrip()
    if len(new_s) > 0:
        prose_texts.append(new_s)

len(prose_texts)

In [ ]:
prose_texts = prose_texts[:50]

prose_texts

In [ ]:
with open("../../corpora/ru_gazeta/processed_gazeta.txt", 'r') as f:
    gazeta = f.readlines()

len(gazeta)

In [ ]:
news_texts = list()
for text in gazeta[:30]:
    for word in word_tokenize(text.lower()):
        if morph.parse(word)[0].normal_form in stopwords:
            break
    else:
        news_texts.append(text)

In [ ]:
len(news_texts)

In [ ]:
news_texts = news_texts[:2]

news_texts

In [ ]:
texts = forum_texts + news_texts + prose_texts

In [ ]:
len(texts)

In [ ]:
tokenizer = nltk.data.load('tokenizers/punkt/russian.pickle')

In [ ]:
nlp = spacy.load("ru_core_news_sm")

In [ ]:
def split_into_clauses_spacy(sentence):
    """Splits a sentence into clauses using spaCy's dependency parser."""
    doc = nlp(sentence)
    clauses = []
    start = 0
    for i, token in enumerate(doc):
        # Detect clause boundaries based on syntactic relations (e.g., conjunctions, subordinating conjunctions)
        if token.dep_ == "cc" or token.dep_ == "mark": # cc: coordinating conjunction, mark: marker (subordinating conjunction)
            clauses.append(doc[start:token.i].text)
            start = token.i
    clauses.append(doc[start:].text)
    return [clause.strip() for clause in clauses if clause.strip()]

In [ ]:
example = """Для данного текста: "Она быстро бегает."

Слова и их разметка могут выглядеть следующим образом:

[
  {"word": "Она", "fee": false, "frame": "-"},
  {"word": "быстро", "fee": true, "frame": "Speed_description"}
  {"word": "бегает", "fee": true, "frame": "Self_motion"},
]
"""

In [ ]:
PROMPT = """ЗАДАНИЕ: Аннотируй данный ТЕКСТ слово за словом с помощью фреймов Berkeley FrameNet из предоставленного списка ФРЕЙМОВ.

ИНСТРУКЦИИ:
1. Используя теорию семантики фреймов Филлмора, проанализируй каждое слово в ТЕКСТЕ с учетом его контекста.
2. Для каждого слова присвой параметры в формате массива JSON: «word», «fee» и «frame».
3. Установи «fee» в TRUE, если слово вызывает или отсылается к фрейму из FrameNet (например, глаголы, обозначающие действия, существительные, обозначающие сущности или концепты, связанные с фреймами). В противном случае установи «fee» в FALSE.
4. Если «fee» равно TRUE, то для параметра «frame» либо выбери подходящий фрейм из ФРЕЙМОВ, либо установите его значение «To_Review», что означает, что в ФРЕЙМАХ нет подходящих фреймов. Ты можешь использовать только фреймы из списка ФРЕЙМЫ.
5. Если слово является наречием, найди существующий фрейм FrameNet, который оно вызывает, и назначь его. Если в списке ФРЕЙМЫ нет подходящих фреймов, установи «frame» равным «To_Review».
6. Если слово является именем собственным, то установи «fee» в FALSE.
7. Если слово является вспомогательным глаголом (например, был, будет, является), то установи «fee» в FALSE.
8. Если «fee» равно FALSE, то установи «frame» равным «-».
9. Предоставь один объект массива JSON со всеми ключами «word», «fee» и «frame», имеющими соответствующие значения.
10. Выведи только объект массива JSON, без объяснений и комментариев.
11. Не пересматривай ответ и не генерируй несколько ответов.

ПРИМЕР:
{example}

ТЕКСТ: {text}

ФРЕЙМЫ: {bfn_frames}
"""

In [ ]:
client = OpenAI(
  api_key=""
)

In [ ]:
responses = list()
all_sentences = list()
for text in tqdm(texts, position=0):
    sentences = tokenizer.tokenize(text)
    for sent in sentences:
        subsentences = split_into_clauses_spacy(sent)
        for subsent in subsentences:
            all_sentences.append(subsent)

            text_prompt = PROMPT.format(example=example,
                                        text=subsent,
                                        bfn_frames=''.join(frames))
            completion = client.chat.completions.create(
                model="gpt-4.1-mini-2025-04-14",
                store=True,
                messages=[
                    {
                        "role": "user",
                        "content": text_prompt
                    }
                ]
            )

            response = completion.choices[0].message.content
            responses.append(response)
            
            # break

In [ ]:
print(responses[0])

In [ ]:
len(all_sentences)

In [ ]:
len(responses)

In [ ]:
df_dict = {"sents": all_sentences, "preds": responses}
df = pd.DataFrame(df_dict)

df

In [ ]:
df.to_csv("../../outputs/fee_annotation/ru_62_gpt_preds_subsentences_adverbs.csv", index=False)